# 02 容器与编排 (Containers & Orchestration)

**课程模块**: 云 & 基础设施 — Senior Data Engineer 面试备考

## 本节覆盖考点

| 技术 | 考点 | 频率 |
|------|------|------|
| Docker | 多阶段构建 & Layer 缓存 | 重要 |
| Kubernetes | Pod / Deployment / Service / PVC | 重要 |
| Kubernetes | Resource Request vs Limit | 重要 |
| Helm | Chart 基础结构 | 了解 |

---

> **学习目标**: 理解容器化数据工程的核心组件，能够设计 Spark-on-K8s / Airflow-on-K8s 的资源配置方案。

---
# Part 1: Docker

## 1.1 Docker 分层存储与 Layer 缓存

Docker 镜像由多个**只读层（Layer）**叠加组成，每条 Dockerfile 指令产生一层。构建时，如果某层的内容（指令+上下文文件哈希）与缓存完全一致，则直接复用缓存，跳过重新构建。

```
Docker Layer 叠加示意:

  ┌─────────────────────────────────────────┐  ← 容器可写层 (Container Layer)
  ├─────────────────────────────────────────┤
  │  COPY src/ /app/src/         (Layer 5)  │  ← 代码 (频繁变化)
  ├─────────────────────────────────────────┤
  │  COPY requirements.txt /app/ (Layer 4)  │  ← 依赖声明 (偶尔变化)
  ├─────────────────────────────────────────┤
  │  RUN pip install -r req.txt  (Layer 3)  │  ← 安装依赖 (慢，但缓存)
  ├─────────────────────────────────────────┤
  │  RUN apt-get install ...     (Layer 2)  │  ← 系统依赖 (很少变化)
  ├─────────────────────────────────────────┤
  │  FROM python:3.11-slim       (Layer 1)  │  ← 基础镜像 (几乎不变)
  └─────────────────────────────────────────┘

Layer 缓存规则:
  某层失效 → 该层及其所有后续层全部重新构建
  因此: 变化频率低的指令放前面，变化频率高的放后面
```

**缓存失效场景**:
- `COPY` 或 `ADD` 的源文件内容改变
- `RUN` 指令文本改变
- 使用 `--no-cache` 强制重建

## 1.2 多阶段构建 (Multi-Stage Build)

**问题**: 构建工具（编译器、测试框架、开发依赖）不应打包进生产镜像，会造成镜像体积膨胀和安全风险。

**解决方案**: 多阶段构建——用多个 `FROM` 指令，最终只保留必要产物。

```
单阶段 vs 多阶段构建对比:

  单阶段:
  ┌────────────────────────────────────┐
  │ python:3.11 (500MB)                │
  │ + build-essential, gcc (200MB)     │
  │ + dev dependencies (300MB)         │
  │ + prod dependencies (150MB)        │
  │ + source code (10MB)               │
  │ = 最终镜像 ~1.2GB                  │
  └────────────────────────────────────┘

  多阶段:
  ┌────────────────────────────────────┐  Build Stage
  │ python:3.11 + build tools          │  (丢弃)
  │ 编译、安装依赖、运行测试            │
  └────────────────────────────────────┘
           │ COPY --from=builder
           ▼ (只拷贝最终产物)
  ┌────────────────────────────────────┐  Runtime Stage
  │ python:3.11-slim (60MB)            │  (保留)
  │ + 已安装的 prod dependencies       │
  │ + source code                      │
  │ = 最终镜像 ~220MB                  │
  └────────────────────────────────────┘
```

In [ ]:
# Docker 多阶段构建示例 - 数据工程 Python 应用

dockerfile_naive = """
# ============================================================
# 低效 Dockerfile (反例): 代码放在依赖安装之前
# ============================================================
FROM python:3.11-slim

WORKDIR /app

# 问题1: 代码先复制，代码变化导致后续所有层缓存失效
COPY . /app/

# 问题2: pip install 在代码之后 → 每次改代码都重装依赖（慢！）
RUN pip install -r requirements.txt

# 问题3: 没有 .dockerignore，连 .git、notebooks 都打包进去了
CMD ["python", "pipeline.py"]
"""

dockerfile_optimized = """
# ============================================================
# 优化 Dockerfile: 多阶段构建 + 正确的 Layer 顺序
# ============================================================

# -------- Stage 1: Build (构建+测试阶段) --------
FROM python:3.11 AS builder

# 安装编译工具（仅构建阶段需要）
RUN apt-get update && apt-get install -y --no-install-recommends \\
        build-essential \\
        libpq-dev \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# 优化点: 先复制依赖文件，再安装 → requirements.txt 不变时缓存命中
COPY requirements.txt requirements-dev.txt ./
RUN pip install --no-cache-dir --user -r requirements.txt -r requirements-dev.txt

# 复制源码（变化最频繁，放最后）
COPY src/ ./src/
COPY tests/ ./tests/

# 运行测试（构建失败则不生成运行时镜像）
RUN python -m pytest tests/ -x -q

# -------- Stage 2: Runtime (运行时阶段) --------
FROM python:3.11-slim AS runtime

# 运行时仅需少量系统库
RUN apt-get update && apt-get install -y --no-install-recommends \\
        libpq5 \\
    && rm -rf /var/lib/apt/lists/* \\
    && useradd -m -r appuser  # 非 root 用户运行（安全最佳实践）

WORKDIR /app

# 只从 builder 阶段复制已安装的依赖（不含 dev 工具）
COPY --from=builder /root/.local /home/appuser/.local

# 复制源码
COPY --from=builder /app/src ./src/

# 切换到非 root 用户
USER appuser

# 健康检查
HEALTHCHECK --interval=30s --timeout=10s CMD python -c "import src; src.health_check()"

ENV PYTHONPATH=/app
CMD ["python", "-m", "src.pipeline"]
"""

dockerignore_content = """
# .dockerignore - 排除不需要的文件（加速构建 + 减小上下文）
.git
.gitignore
__pycache__/
*.pyc
*.egg-info/
.env
.env.*
*.ipynb
.ipynb_checkpoints/
tests/
docs/
README.md
.pytest_cache/
.mypy_cache/
dist/
build/
*.log
node_modules/
"""

print("=== 低效 Dockerfile (反例) ===")
print(dockerfile_naive)
print("\n=== 优化 Dockerfile (多阶段 + 正确 Layer 顺序) ===")
print(dockerfile_optimized)
print("\n=== .dockerignore ===")
print(dockerignore_content)

In [ ]:
# Layer 缓存效率模拟

def simulate_build_time(scenario: str, layers: list) -> dict:
    """
    Simulate Docker build time with and without cache.
    layers: list of (name, build_time_sec, cache_hit)
    """
    total_time = 0
    cache_saved = 0
    details = []
    cache_broken = False

    for layer_name, build_time_sec, would_cache in layers:
        if cache_broken:
            # Once cache is broken, all subsequent layers must rebuild
            actual_time = build_time_sec
            status = 'REBUILD (cache broken by previous layer)'
        elif would_cache:
            actual_time = 0.1  # cache hit ~100ms
            cache_saved += build_time_sec - actual_time
            status = 'CACHED'
        else:
            actual_time = build_time_sec
            cache_broken = True  # This layer breaks the cache
            status = 'REBUILD (changed)'

        total_time += actual_time
        details.append({'layer': layer_name, 'time_sec': actual_time, 'status': status})

    return {
        'scenario': scenario,
        'total_build_time_sec': round(total_time, 1),
        'cache_time_saved_sec': round(cache_saved, 1),
        'layers': details
    }


# 场景1: 低效 Dockerfile — 代码在依赖前，代码变化时 pip install 重跑
naive_layers = [
    ('FROM python:3.11-slim',      2.0,  True),
    ('COPY . /app/',               1.0,  False),   # 代码变化 → 缓存失效
    ('RUN pip install ...',       90.0,  False),   # 必须重新安装所有依赖
]

# 场景2: 优化 Dockerfile — 只有代码层重建，pip install 缓存命中
optimized_layers = [
    ('FROM python:3.11-slim',      2.0,  True),
    ('COPY requirements.txt',      0.5,  True),    # requirements 没变 → 缓存
    ('RUN pip install ...',       90.0,  True),    # 缓存命中！
    ('COPY src/ /app/src/',        1.0,  False),   # 代码变化
]

result_naive = simulate_build_time('Naive Dockerfile', naive_layers)
result_optimized = simulate_build_time('Optimized Dockerfile', optimized_layers)

for result in [result_naive, result_optimized]:
    print(f"\n=== {result['scenario']} ===")
    print(f"  Total build time: {result['total_build_time_sec']}s")
    print(f"  Cache time saved: {result['cache_time_saved_sec']}s")
    print("  Layer details:")
    for layer in result['layers']:
        print(f"    [{layer['status']:<40}] {layer['layer']:<35} ({layer['time_sec']}s)")

print(f"\n>>> Speedup: {result_naive['total_build_time_sec']}s vs {result_optimized['total_build_time_sec']}s")
print(f">>> Cache optimization saves {result_naive['total_build_time_sec'] - result_optimized['total_build_time_sec']:.1f}s per build")

---
# Part 2: Kubernetes 核心概念

## 2.1 核心对象层级

```
Kubernetes 对象层级 (从上到下):

  Namespace (逻辑隔离边界)
  └── Deployment (声明式管理 Pod 副本数 + 滚动更新策略)
      └── ReplicaSet (维持指定数量的 Pod 副本)
          └── Pod (最小调度单位，包含1个或多个容器)
              └── Container (运行的 Docker 镜像)

  Service (网络访问抽象，稳定的 ClusterIP/DNS)
  └── 通过 Label Selector 匹配 Pod

  PersistentVolume (PV) → 管理员预先配置的存储
  PersistentVolumeClaim (PVC) → 用户申请存储的方式
  StorageClass → 动态 PV 配置模板
```

## 2.2 Service 类型

| Service 类型 | 访问范围 | 典型用途 |
|-------------|---------|----------|
| **ClusterIP** | 集群内部 | 服务间通信（Airflow Scheduler → Worker）|
| **NodePort** | 节点 IP + 端口 | 开发/测试环境外部访问 |
| **LoadBalancer** | 外部负载均衡 IP | 生产环境用户访问（Airflow WebServer）|
| **ExternalName** | DNS CNAME | 访问集群外部服务 |

## 2.3 Airflow KubernetesExecutor 工作原理

```
Airflow KubernetesExecutor 架构:

  ┌─────────────────────────────────────────────┐
  │           Airflow Scheduler Pod             │
  │  (调度器：解析 DAG，决定哪些任务运行)         │
  └──────────────────┬──────────────────────────┘
                     │ 调用 K8s API 创建 Pod
                     ▼
  ┌──────────────────────────────────────────────┐
  │               Kubernetes API Server          │
  └──────────────────┬───────────────────────────┘
                     │ 调度到节点
     ┌───────────────┼───────────────┐
     ▼               ▼               ▼
  ┌──────┐       ┌──────┐       ┌──────┐
  │Task  │       │Task  │       │Task  │
  │Pod 1 │       │Pod 2 │       │Pod 3 │  ← 每个任务独立 Pod
  │(ETL) │       │(ML)  │       │(BQ)  │    任务完成 → Pod 销毁
  └──────┘       └──────┘       └──────┘

优势: 完美资源隔离，每个任务按需申请资源
劣势: Pod 冷启动时间 (2-10秒)，不适合高频短任务
```

In [ ]:
# Kubernetes YAML 配置示例（数据工程场景）

pod_yaml = """
# Pod YAML: 最小可运行单元
apiVersion: v1
kind: Pod
metadata:
  name: spark-driver
  namespace: data-engineering
  labels:
    app: spark
    role: driver
    team: data-eng
spec:
  serviceAccountName: spark-sa  # 允许 driver 调用 K8s API 创建 executor pods
  containers:
  - name: spark-driver
    image: apache/spark:3.5.0
    args: ["driver", "--class", "com.example.ETLJob"]
    resources:
      requests:           # 调度依据: 节点必须有这么多可用资源才能调度
        cpu: "2"          # 2 CPU cores
        memory: "4Gi"     # 4 GB RAM
      limits:             # 硬上限: 超过则 CPU 限速 / 内存 OOM kill
        cpu: "4"
        memory: "6Gi"
    env:
    - name: SPARK_CONF_DIR
      value: /opt/spark/conf
    - name: AWS_ACCESS_KEY_ID
      valueFrom:
        secretKeyRef:     # 从 K8s Secret 读取，不硬编码在 YAML
          name: aws-credentials
          key: access-key-id
    volumeMounts:
    - name: spark-config
      mountPath: /opt/spark/conf
  volumes:
  - name: spark-config
    configMap:            # 从 ConfigMap 挂载配置文件
      name: spark-config
  restartPolicy: Never    # Spark driver 失败不重启（重跑由外部调度器控制）
"""

deployment_yaml = """
# Deployment YAML: Airflow Scheduler 示例
apiVersion: apps/v1
kind: Deployment
metadata:
  name: airflow-scheduler
  namespace: airflow
spec:
  replicas: 2                      # 高可用: 2个调度器 (active-passive)
  selector:
    matchLabels:
      app: airflow
      component: scheduler
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1                  # 更新时最多多出 1 个新 Pod
      maxUnavailable: 1            # 更新时最多允许 1 个 Pod 不可用
  template:
    metadata:
      labels:
        app: airflow
        component: scheduler
    spec:
      containers:
      - name: scheduler
        image: apache/airflow:2.9.0
        command: ["airflow", "scheduler"]
        resources:
          requests:
            cpu: "500m"            # 0.5 CPU core (m = millicores)
            memory: "2Gi"
          limits:
            cpu: "2"
            memory: "4Gi"
        livenessProbe:             # 存活探针: 失败则重启容器
          exec:
            command: ["airflow", "jobs", "check", "--job-type", "SchedulerJob"]
          initialDelaySeconds: 30
          periodSeconds: 60
        readinessProbe:            # 就绪探针: 失败则从 Service 摘除
          httpGet:
            path: /health
            port: 8974
          initialDelaySeconds: 20
          periodSeconds: 10
"""

service_yaml = """
# Service YAML: 为 Airflow WebServer 提供负载均衡访问
apiVersion: v1
kind: Service
metadata:
  name: airflow-webserver
  namespace: airflow
spec:
  type: LoadBalancer              # 在云环境创建外部负载均衡器 (ALB/NLB)
  selector:
    app: airflow
    component: webserver          # 匹配带有这些 label 的 Pod
  ports:
  - name: http
    port: 80
    targetPort: 8080
  - name: https
    port: 443
    targetPort: 8080
"""

pvc_yaml = """
# PersistentVolumeClaim: Airflow DAG 文件持久存储
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: airflow-dags-pvc
  namespace: airflow
spec:
  accessModes:
  - ReadWriteMany              # RWX: 多个 Pod 可同时读写 (需要 NFS/EFS StorageClass)
  storageClassName: efs-sc     # AWS EFS StorageClass
  resources:
    requests:
      storage: 10Gi            # 申请 10GB 存储
"""

yaml_examples = [
    ('Pod (Spark Driver)', pod_yaml),
    ('Deployment (Airflow Scheduler)', deployment_yaml),
    ('Service (Airflow WebServer)', service_yaml),
    ('PersistentVolumeClaim (DAG Storage)', pvc_yaml)
]

for title, yaml_content in yaml_examples:
    print(f"\n{'='*60}")
    print(f"### {title} ###")
    print('='*60)
    print(yaml_content)

## 2.3 Resource Request vs Limit — 重要区别

这是面试高频考点，必须清楚两者的区别：

### CPU: Request vs Limit

```
CPU Throttling 机制:

  Request = 500m (0.5 core)
  Limit   = 2000m (2 cores)

  节点有空闲 CPU 时:
  [████████████████████] 容器可以突发到 2 cores

  节点 CPU 紧张时:
  [██████████░░░░░░░░░░] 容器被限速到 500m
   (按 Request 比例分配空闲 CPU)

  超过 Limit:
  [被 throttle，不会被 kill，只是变慢]
```

### Memory: Request vs Limit

```
Memory OOM Kill 机制:

  Request = 2Gi
  Limit   = 4Gi

  内存使用 < 4Gi: 正常运行

  内存使用 ≥ 4Gi:
  ┌─────────────────────────────────────┐
  │  OOM Killed! (内核直接杀掉进程)      │
  │  Exit Code: 137 (SIGKILL)           │
  │  Pod 状态: OOMKilled                 │
  └─────────────────────────────────────┘
  → 内存不像 CPU 可以 throttle，超限直接 Kill
```

### QoS (Quality of Service) 等级

K8s 根据 Request/Limit 配置将 Pod 分为 3 个 QoS 等级，资源紧张时驱逐顺序不同：

| QoS 等级 | 条件 | 驱逐优先级 |
|---------|------|----------|
| **Guaranteed** | Request = Limit (两者相等) | 最后驱逐 |
| **Burstable** | Request < Limit | 中间 |
| **BestEffort** | 未设置 Request 和 Limit | 最先驱逐 |

> **面试答点**: 生产环境关键服务（Airflow Scheduler）设置 Guaranteed QoS；批处理任务设置 Burstable 以提高资源利用率。

In [ ]:
# Spark on K8s 资源规划计算

def spark_on_k8s_sizing(
    total_node_count: int,
    node_cpu_cores: int,
    node_memory_gb: int,
    system_reserved_cpu_cores: float = 1.0,
    system_reserved_memory_gb: float = 2.0,
    executor_cpu_cores: int = 2,
    executor_memory_gb: int = 8,
    executor_memory_overhead_gb: float = 2.0,  # spark.executor.memoryOverhead
    driver_cpu_cores: int = 1,
    driver_memory_gb: int = 4
) -> dict:
    """Calculate optimal Spark on Kubernetes resource configuration."""

    # Allocatable resources per node (after system reservation)
    allocatable_cpu = node_cpu_cores - system_reserved_cpu_cores
    allocatable_memory_gb = node_memory_gb - system_reserved_memory_gb

    # Total allocatable across all nodes
    total_allocatable_cpu = allocatable_cpu * total_node_count
    total_allocatable_memory_gb = allocatable_memory_gb * total_node_count

    # Memory per executor (including overhead)
    executor_total_memory = executor_memory_gb + executor_memory_overhead_gb

    # Max executors limited by CPU
    max_executors_by_cpu = int(total_allocatable_cpu / executor_cpu_cores)

    # Max executors limited by memory
    max_executors_by_memory = int(
        (total_allocatable_memory_gb - driver_memory_gb) / executor_total_memory
    )

    # Actual max executors (bottleneck resource)
    max_executors = min(max_executors_by_cpu, max_executors_by_memory)
    bottleneck = 'CPU' if max_executors_by_cpu < max_executors_by_memory else 'Memory'

    # Recommended: leave 20% headroom for K8s overhead
    recommended_executors = int(max_executors * 0.8)

    return {
        'cluster_config': {
            'nodes': total_node_count,
            'cpu_per_node': node_cpu_cores,
            'memory_gb_per_node': node_memory_gb,
            'allocatable_cpu_per_node': allocatable_cpu,
            'allocatable_memory_gb_per_node': allocatable_memory_gb,
        },
        'spark_config': {
            'executor_cpu_cores': executor_cpu_cores,
            'executor_memory_gb': executor_memory_gb,
            'executor_memory_overhead_gb': executor_memory_overhead_gb,
            'executor_total_memory_gb': executor_total_memory,
        },
        'results': {
            'max_executors_by_cpu': max_executors_by_cpu,
            'max_executors_by_memory': max_executors_by_memory,
            'actual_max_executors': max_executors,
            'bottleneck_resource': bottleneck,
            'recommended_executors': recommended_executors,
            'spark_submit_args': (
                f'--num-executors {recommended_executors} '
                f'--executor-cores {executor_cpu_cores} '
                f'--executor-memory {executor_memory_gb}g '
                f'--conf spark.executor.memoryOverhead={executor_memory_overhead_gb}g'
            )
        }
    }


# 场景: 10节点集群，每节点 16 CPU / 64GB RAM
result = spark_on_k8s_sizing(
    total_node_count=10,
    node_cpu_cores=16,
    node_memory_gb=64,
    executor_cpu_cores=2,
    executor_memory_gb=8,
    executor_memory_overhead_gb=2
)

print("=" * 60)
print("Spark on K8s Resource Sizing Calculator")
print("=" * 60)

for section, values in result.items():
    print(f"\n[{section.upper()}]")
    for k, v in values.items():
        print(f"  {k}: {v}")

print()
print("Kubernetes YAML snippet for executor pods:")
executor_resources_yaml = f"""
  resources:
    requests:
      cpu: "{result['spark_config']['executor_cpu_cores']}"
      memory: "{result['spark_config']['executor_total_memory_gb']}Gi"
    limits:
      cpu: "{result['spark_config']['executor_cpu_cores']}"
      memory: "{result['spark_config']['executor_total_memory_gb']}Gi"
  # Note: Request == Limit → Guaranteed QoS (executor is eviction-resistant)
"""
print(executor_resources_yaml)

In [ ]:
# LimitRange & ResourceQuota 配置示例

limitrange_yaml = """
# LimitRange: 为 Namespace 设置默认/最大 Resource 限制
# 防止用户忘记设置 limits 导致资源无限占用
apiVersion: v1
kind: LimitRange
metadata:
  name: data-eng-limits
  namespace: data-engineering
spec:
  limits:
  - type: Container
    default:              # 未设置 Limit 时的默认值
      cpu: "1"
      memory: 2Gi
    defaultRequest:       # 未设置 Request 时的默认值
      cpu: "250m"
      memory: 512Mi
    max:                  # 单个容器允许的最大值
      cpu: "16"
      memory: 64Gi
    min:                  # 单个容器允许的最小值
      cpu: "50m"
      memory: 64Mi
"""

resourcequota_yaml = """
# ResourceQuota: 限制整个 Namespace 的资源总量
# 防止一个团队耗尽集群资源
apiVersion: v1
kind: ResourceQuota
metadata:
  name: data-eng-quota
  namespace: data-engineering
spec:
  hard:
    # 计算资源
    requests.cpu: "100"         # Namespace 内所有 Pod 的 CPU Request 总和 <= 100 cores
    requests.memory: 400Gi      # 总 Memory Request <= 400 GB
    limits.cpu: "200"           # 总 CPU Limit <= 200 cores
    limits.memory: 800Gi        # 总 Memory Limit <= 800 GB
    # 对象数量
    pods: "200"                 # 最多 200 个 Pod
    services: "20"
    persistentvolumeclaims: "50"
    # 存储
    requests.storage: 5Ti       # 总 PVC 存储申请量 <= 5 TB
"""

print("=== LimitRange: 容器级别默认值和边界 ===")
print(limitrange_yaml)
print("\n=== ResourceQuota: Namespace 级别总量限制 ===")
print(resourcequota_yaml)

# Resource 设置最佳实践总结
best_practices = {
    '关键服务 (Airflow Scheduler, DB)': {
        'QoS目标': 'Guaranteed',
        'cpu_request': '等于 cpu_limit',
        'memory_request': '等于 memory_limit',
        '理由': '节点资源紧张时不被驱逐'
    },
    '批处理任务 (Spark Executor, ETL)': {
        'QoS目标': 'Burstable',
        'cpu_request': '设为 limit 的 25-50%',
        'memory_request': '设为 limit 的 75-100%',
        '理由': 'CPU 可突发利用空闲；内存不能超 limit 否则 OOM'
    },
    '开发/测试 Pod': {
        'QoS目标': 'BestEffort 或 Burstable',
        'cpu_request': '极小 (50m)',
        'memory_request': '按实际需求低配',
        '理由': '资源紧张时优先驱逐，不影响生产'
    }
}

print("\n=== Resource 设置最佳实践 ===")
for service_type, config in best_practices.items():
    print(f"\n{service_type}:")
    for k, v in config.items():
        print(f"  {k}: {v}")

---
# Part 3: Helm Chart 基础

## 3.1 Helm 是什么？

Helm 是 Kubernetes 的**包管理器**，将一组相关的 K8s YAML 文件打包为 **Chart**，支持版本管理、参数化部署和回滚。

```
Helm Chart 目录结构:

  my-data-pipeline/              ← Chart 根目录
  ├── Chart.yaml                 ← Chart 元数据 (名称、版本、依赖)
  ├── values.yaml                ← 默认参数值 (可被 --set 或 -f 覆盖)
  ├── templates/                 ← K8s YAML 模板文件
  │   ├── _helpers.tpl           ← 公共模板片段
  │   ├── deployment.yaml        ← Deployment 模板
  │   ├── service.yaml           ← Service 模板
  │   ├── configmap.yaml         ← ConfigMap 模板
  │   ├── secret.yaml            ← Secret 模板
  │   └── NOTES.txt              ← 安装后的提示信息
  ├── charts/                    ← 子 Chart 依赖
  └── .helmignore                ← 类似 .gitignore

Helm 工作流:

  values.yaml  +  templates/  →  helm template  →  K8s YAML
                                                         ↓
                                                   kubectl apply
                                                   (helm install/upgrade)
```

## 3.2 常用 Helm 命令

```bash
# 安装 Chart
helm install airflow apache-airflow/airflow -n airflow \
  --create-namespace \
  -f custom-values.yaml          # 覆盖默认值

# 升级
helm upgrade airflow apache-airflow/airflow -n airflow \
  --set scheduler.replicas=2 \
  --atomic                        # 失败自动回滚

# 回滚到上一个版本
helm rollback airflow 1 -n airflow

# 查看历史版本
helm history airflow -n airflow

# 渲染模板（不实际部署，用于 debug）
helm template airflow apache-airflow/airflow -f custom-values.yaml

# 查看所有安装的 Release
helm list -A
```

In [ ]:
# Helm Chart 配置示例（数据管道部署）

chart_yaml = """
# Chart.yaml
apiVersion: v2
name: data-pipeline
description: Data engineering pipeline for batch ETL processing
type: application
version: 1.3.0          # Chart 版本
appVersion: "2.1.0"     # 应用版本
keywords:
  - data-engineering
  - etl
  - spark
dependencies:
  - name: postgresql     # 子 Chart 依赖（Helm Metadata Store）
    version: "12.x.x"
    repository: https://charts.bitnami.com/bitnami
    condition: postgresql.enabled
"""

values_yaml = """
# values.yaml: 默认配置（可被环境特定 values 文件覆盖）
replicaCount: 1

image:
  repository: my-registry.io/data-pipeline
  tag: "2.1.0"
  pullPolicy: IfNotPresent

resources:
  requests:
    cpu: 500m
    memory: 2Gi
  limits:
    cpu: 2000m
    memory: 4Gi

env:
  LOG_LEVEL: INFO
  BATCH_SIZE: "1000"

spark:
  executorInstances: 10
  executorMemory: 8g
  executorCores: 2

postgresql:
  enabled: true
  auth:
    database: metadata_db

autoscaling:
  enabled: false
  minReplicas: 1
  maxReplicas: 5
"""

values_prod_yaml = """
# values-prod.yaml: 生产环境覆盖值
replicaCount: 3                    # 生产高可用

image:
  tag: "2.1.0-stable"              # 固定稳定版本

resources:
  requests:
    cpu: 2000m                     # 生产给更多资源
    memory: 8Gi
  limits:
    cpu: 2000m                     # Guaranteed QoS
    memory: 8Gi

env:
  LOG_LEVEL: WARNING               # 生产减少日志量
  BATCH_SIZE: "5000"               # 生产大批量

spark:
  executorInstances: 50
  executorMemory: 16g
  executorCores: 4

autoscaling:
  enabled: true
  minReplicas: 3
  maxReplicas: 20
"""

deployment_template_yaml = """
# templates/deployment.yaml (Helm 模板语法)
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ include "data-pipeline.fullname" . }}
  labels:
    {{- include "data-pipeline.labels" . | nindent 4 }}
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      {{- include "data-pipeline.selectorLabels" . | nindent 6 }}
  template:
    spec:
      containers:
      - name: {{ .Chart.Name }}
        image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"
        imagePullPolicy: {{ .Values.image.pullPolicy }}
        env:
        {{- range $key, $value := .Values.env }}
        - name: {{ $key }}
          value: {{ $value | quote }}
        {{- end }}
        resources:
          {{- toYaml .Values.resources | nindent 10 }}
"""

examples = [
    ('Chart.yaml', chart_yaml),
    ('values.yaml (defaults)', values_yaml),
    ('values-prod.yaml (overrides)', values_prod_yaml),
    ('templates/deployment.yaml', deployment_template_yaml)
]

for filename, content in examples:
    print(f"\n{'='*60}")
    print(f"### {filename} ###")
    print('='*60)
    print(content)

print("\n=== Deploy Commands ===")
deploy_commands = """
# 开发环境部署
helm upgrade --install data-pipeline ./data-pipeline \\
  -n dev --create-namespace

# 生产环境部署 (使用覆盖值)
helm upgrade --install data-pipeline ./data-pipeline \\
  -n prod \\
  -f values-prod.yaml \\
  --set image.tag=2.1.1 \\
  --atomic \\
  --timeout 10m
"""
print(deploy_commands)

---
# 练习题

## 练习 1: Docker 多阶段构建优化

给定以下 Dockerfile，找出所有问题并重写优化版本：

```dockerfile
FROM python:3.11

WORKDIR /app

# 安装系统依赖
RUN apt-get update && apt-get install -y gcc libpq-dev curl

# 先复制所有文件
COPY . .

# 再安装 Python 依赖
RUN pip install pandas==2.0.0 pyarrow==12.0.0 sqlalchemy==2.0.0 pytest==7.4.0

# 运行测试
RUN pytest tests/

# 启动
CMD ["python", "main.py"]
```

**问题**:
1. 找出至少 4 个问题（Layer 缓存/安全/体积/构建效率）
2. 写出优化后的 Dockerfile（多阶段 + 正确顺序）
3. 写出对应的 .dockerignore 文件内容

In [ ]:
# 练习 1 参考答案

problems = """
问题清单:
1. [Layer 缓存] COPY . . 放在 pip install 之前 → 任何源码修改都导致 pip install 重跑
2. [安全] 使用 root 用户运行容器 → 生产安全风险
3. [体积] 使用 python:3.11 全量镜像而非 python:3.11-slim → 镜像偏大
4. [体积] 测试工具 (pytest) 打包进生产镜像 → 应多阶段构建分离
5. [体积] apt 缓存未清理 → rm -rf /var/lib/apt/lists/* 缺失
6. [效率] pip 列在行内而非 requirements.txt → 难以管理版本和利用缓存
"""

optimized_dockerfile = """
# ===== 优化后的 Dockerfile =====

# Stage 1: Build & Test
FROM python:3.11-slim AS builder

RUN apt-get update && apt-get install -y --no-install-recommends \\
        build-essential gcc libpq-dev \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# 先复制依赖文件 (变化频率低) → pip install 可缓存
COPY requirements.txt requirements-dev.txt ./
RUN pip install --no-cache-dir --user \\
        -r requirements.txt \\
        -r requirements-dev.txt

# 再复制源码 (变化频率高)
COPY src/ ./src/
COPY tests/ ./tests/

# 测试通过才继续
RUN python -m pytest tests/ -x -q

# Stage 2: Runtime (精简)
FROM python:3.11-slim AS runtime

RUN apt-get update && apt-get install -y --no-install-recommends \\
        libpq5 \\
    && rm -rf /var/lib/apt/lists/* \\
    && useradd -m -r appuser  # 非 root

WORKDIR /app

# 只复制已安装的 prod 依赖（不包含 dev 工具如 pytest）
COPY --from=builder /root/.local /home/appuser/.local
COPY --from=builder /app/src ./src/

USER appuser
ENV PYTHONPATH=/app PATH=/home/appuser/.local/bin:$PATH
CMD ["python", "-m", "src.main"]
"""

dockerignore = """
# .dockerignore
.git/
.gitignore
__pycache__/
*.pyc
.env
.env.*
*.ipynb
.ipynb_checkpoints/
tests/           # tests 只在 builder stage 需要，通过 COPY tests/ 显式引入
docs/
*.md
.pytest_cache/
dist/
build/
"""

print("=== 练习 1 参考答案 ===")
print(problems)
print("\n--- 优化后 Dockerfile ---")
print(optimized_dockerfile)
print("\n--- .dockerignore ---")
print(dockerignore)

## 练习 2: K8s Resource 设计

你要在 K8s 上部署以下数据工程服务，为每个服务设计 Resource Request 和 Limit：

| 服务 | 描述 | 特点 |
|------|------|------|
| Airflow Scheduler | 核心调度服务 | 不能被驱逐，CPU 用量稳定 |
| Spark Executor | 批处理计算 | 可以被重新调度，内存敏感 |
| Jupyter Notebook | 数据探索 | 开发环境，非关键 |
| DBT Runner | SQL 转换任务 | 间歇性运行，内存中等 |

集群规格：10 个节点，每节点 16 CPU / 64GB RAM。

**问题**:
1. 为每个服务设置合适的 Request 和 Limit
2. 说明每个服务的 QoS 等级
3. 写出 Namespace ResourceQuota（假设4个服务最多同时运行50个Pod）

In [ ]:
# 练习 2 参考答案

resource_designs = [
    {
        'service': 'Airflow Scheduler',
        'cpu_request': '1000m',
        'cpu_limit': '1000m',
        'memory_request': '4Gi',
        'memory_limit': '4Gi',
        'qos': 'Guaranteed (Request == Limit)',
        'rationale': '核心调度服务不能被驱逐，使用 Guaranteed QoS。CPU 用量稳定，Request=Limit 避免throttle'
    },
    {
        'service': 'Spark Executor',
        'cpu_request': '2000m',
        'cpu_limit': '2000m',
        'memory_request': '10Gi',  # executor_memory(8g) + overhead(2g)
        'memory_limit': '10Gi',
        'qos': 'Guaranteed (Request == Limit, 内存特别重要)',
        'rationale': '内存超限会 OOM Kill 导致任务失败。Spark Executor 对内存敏感，建议 Guaranteed。CPU 可等于 Limit'
    },
    {
        'service': 'Jupyter Notebook',
        'cpu_request': '250m',
        'cpu_limit': '4000m',
        'memory_request': '1Gi',
        'memory_limit': '8Gi',
        'qos': 'Burstable (Request < Limit)',
        'rationale': '开发环境非关键，资源紧张时可被驱逐。允许突发使用多 CPU/内存提升交互体验'
    },
    {
        'service': 'DBT Runner',
        'cpu_request': '500m',
        'cpu_limit': '2000m',
        'memory_request': '2Gi',
        'memory_limit': '4Gi',
        'qos': 'Burstable',
        'rationale': '间歇性运行，不需要持续占用高资源。允许 CPU 突发处理复杂模型编译'
    }
]

print("=== 练习 2 参考答案: K8s Resource 设计 ===")
print()
for svc in resource_designs:
    print(f"Service: {svc['service']}")
    print(f"  CPU:    Request={svc['cpu_request']}, Limit={svc['cpu_limit']}")
    print(f"  Memory: Request={svc['memory_request']}, Limit={svc['memory_limit']}")
    print(f"  QoS:    {svc['qos']}")
    print(f"  理由:   {svc['rationale']}")
    print()

resource_quota_answer = """
apiVersion: v1
kind: ResourceQuota
metadata:
  name: data-eng-quota
  namespace: data-engineering
spec:
  hard:
    # 计算: 50个 Pod 最大资源估算
    # Scheduler(2) + Executor(30) + Jupyter(5) + DBT(13) = 50 pods
    # CPU requests: 2*1 + 30*2 + 5*0.25 + 13*0.5 = 70.75 cores
    # Memory requests: 2*4 + 30*10 + 5*1 + 13*2 = 369 Gi
    requests.cpu: "80"      # 留 15% 余量
    requests.memory: 420Gi
    limits.cpu: "160"       # Limit 是 Request 的约 2x
    limits.memory: 700Gi
    pods: "50"
"""
print("\n=== Namespace ResourceQuota ===")
print(resource_quota_answer)

## 练习 3: 架构设计 — Airflow on K8s

设计一个生产级别的 Airflow on Kubernetes 部署架构，要求：
- 高可用（Scheduler 不能单点）
- DAG 文件需要在所有组件间共享
- WebServer 需要外部访问
- 不同任务需要不同的资源配置

**问题**: 画出架构图（ASCII），列出所需的 K8s 对象，并说明每个设计决策。

In [ ]:
# 练习 3 参考答案

architecture = """
=== Airflow on K8s 生产架构 (ASCII) ===

  External Users
       │
       ▼
  ┌─────────────────────────────────────────────────────────────┐
  │  LoadBalancer Service (port 443)                            │
  └─────────────────────────┬───────────────────────────────────┘
                            │
  ┌─────────────────────────▼───────────────────────────────────┐
  │  Deployment: airflow-webserver (replicas=2)                 │
  │  Resource: Request=0.5CPU/1Gi, Limit=2CPU/2Gi (Burstable)  │
  └─────────────────────────┬───────────────────────────────────┘
                            │ (ClusterIP Service)
  ┌─────────────────────────▼───────────────────────────────────┐
  │  Deployment: airflow-scheduler (replicas=2)                 │
  │  Resource: Request=Limit=1CPU/4Gi (Guaranteed QoS)          │
  │  LivenessProbe: airflow jobs check --job-type SchedulerJob  │
  └─────────────────────────┬───────────────────────────────────┘
                            │ 调用 K8s API 创建 Task Pods
  ┌─────────────────────────▼───────────────────────────────────┐
  │  K8s Task Pods (KubernetesExecutor)                         │
  │  ┌──────────────┐  ┌──────────────┐  ┌──────────────────┐  │
  │  │ ETL Task Pod │  │ Spark Task   │  │ BigQuery Task   │  │
  │  │ 1CPU/2Gi     │  │ 4CPU/16Gi    │  │ 0.5CPU/1Gi      │  │
  │  └──────────────┘  └──────────────┘  └──────────────────┘  │
  └─────────────────────────────────────────────────────────────┘

  共享存储层:
  ┌─────────────────────────────────────────────────────────────┐
  │  PVC: airflow-dags (ReadWriteMany, AWS EFS, 10Gi)           │
  │  挂载到: webserver, scheduler, ALL task pods                 │
  └─────────────────────────────────────────────────────────────┘

  配置和凭证:
  ┌─────────────────────────────────────────────────────────────┐
  │  ConfigMap: airflow-config (airflow.cfg, connections)       │
  │  Secret: airflow-fernet-key, aws-credentials, db-password   │
  └─────────────────────────────────────────────────────────────┘

  元数据数据库:
  ┌─────────────────────────────────────────────────────────────┐
  │  StatefulSet: postgresql / RDS (推荐外部托管)               │
  │  ClusterIP Service: airflow-postgresql:5432                 │
  └─────────────────────────────────────────────────────────────┘
"""

k8s_objects = """
所需 K8s 对象清单:

Namespace:
  - airflow                          # 专用命名空间

Deployments:
  - airflow-webserver (replicas=2)   # 高可用 WebUI
  - airflow-scheduler (replicas=2)   # 高可用调度器
  - airflow-triggerer  (replicas=1)  # Deferrable Operator 触发器

Services:
  - airflow-webserver (LoadBalancer) # 外部访问入口
  - airflow-postgresql (ClusterIP)   # 内部数据库访问

Storage:
  - PVC: airflow-dags (ReadWriteMany, EFS StorageClass)
  - PVC: airflow-logs (ReadWriteMany, EFS StorageClass)

Config/Secret:
  - ConfigMap: airflow-config
  - Secret: airflow-fernet-key
  - Secret: airflow-metadata-db-credentials
  - Secret: airflow-connections

RBAC (KubernetesExecutor 权限):
  - ServiceAccount: airflow-scheduler
  - Role: airflow-pod-launcher       # 允许创建/删除 Pod
  - RoleBinding: scheduler-pod-launcher

Resource Governance:
  - LimitRange: airflow-limits
  - ResourceQuota: airflow-quota
"""

design_decisions = """
设计决策说明:

1. Scheduler replicas=2:
   - Airflow 2.x 支持多 Scheduler（HA 模式）
   - 使用 Guaranteed QoS 防止资源紧张时被驱逐

2. DAG 存储用 ReadWriteMany PVC (EFS):
   - Scheduler 和所有 Task Pod 需要访问同一份 DAG 文件
   - ReadWriteOnce 只允许单个节点挂载，不够用
   - AWS EFS 支持 RWX，GCP 可用 Filestore

3. KubernetesExecutor vs CeleryExecutor:
   - KubernetesExecutor: 每个任务独立 Pod，资源隔离强，适合异构任务
   - CeleryExecutor: Worker 常驻，适合高频短任务（冷启动慢是 K8s 的劣势）

4. Task Pod 资源用 pod_override:
   @task(executor_config={'pod_override': V1Pod(spec=V1PodSpec(
       containers=[V1Container(name='base', resources=V1ResourceRequirements(
           requests={'cpu': '4', 'memory': '16Gi'},
           limits={'cpu': '4', 'memory': '16Gi'}
       ))]
   ))})
   def heavy_spark_task(): ...
"""

print(architecture)
print(k8s_objects)
print(design_decisions)

## 练习 4: Helm Values 参数化

你需要用 Helm 管理一个数据管道在 dev/staging/prod 三个环境的部署。

已知各环境差异：
- **dev**: 1个副本，小资源，日志 DEBUG，连接测试数据库
- **staging**: 2个副本，中等资源，日志 INFO，连接 staging 数据库
- **prod**: 3个副本，大资源，日志 WARNING，连接生产数据库，开启 autoscaling

**问题**: 写出 values.yaml（默认值）和 values-prod.yaml（生产覆盖），以及对应的部署命令。

In [ ]:
# 练习 4 参考答案

values_base = """
# values.yaml (dev 默认值)
replicaCount: 1
environment: dev

image:
  repository: my-registry/data-pipeline
  tag: latest
  pullPolicy: Always

resources:
  requests:
    cpu: 250m
    memory: 512Mi
  limits:
    cpu: 1000m
    memory: 2Gi

config:
  logLevel: DEBUG
  batchSize: 100
  databaseUrl: postgresql://dev-db:5432/pipeline_dev

autoscaling:
  enabled: false
  minReplicas: 1
  maxReplicas: 3
  targetCPUUtilizationPercentage: 80
"""

values_staging = """
# values-staging.yaml
replicaCount: 2
environment: staging

image:
  tag: "1.5.0-rc1"    # 候选版本
  pullPolicy: IfNotPresent

resources:
  requests:
    cpu: 500m
    memory: 1Gi
  limits:
    cpu: 2000m
    memory: 4Gi

config:
  logLevel: INFO
  batchSize: 1000
  databaseUrl: postgresql://staging-db:5432/pipeline_staging
"""

values_prod = """
# values-prod.yaml
replicaCount: 3
environment: prod

image:
  tag: "1.5.0"        # 固定稳定版本
  pullPolicy: IfNotPresent

resources:
  requests:
    cpu: 2000m
    memory: 4Gi
  limits:
    cpu: 2000m        # Guaranteed QoS: Request == Limit
    memory: 4Gi

config:
  logLevel: WARNING
  batchSize: 10000
  databaseUrl: postgresql://prod-db:5432/pipeline_prod  # 实际用 Secret 注入

autoscaling:
  enabled: true
  minReplicas: 3
  maxReplicas: 20
  targetCPUUtilizationPercentage: 70
"""

deploy_commands = """
# Dev 部署
helm upgrade --install pipeline-dev ./data-pipeline \\
  --namespace dev --create-namespace

# Staging 部署
helm upgrade --install pipeline-staging ./data-pipeline \\
  --namespace staging \\
  -f values-staging.yaml

# Prod 部署 (带安全措施)
helm upgrade --install pipeline-prod ./data-pipeline \\
  --namespace prod \\
  -f values-prod.yaml \\
  --set config.databaseUrl='' \\
  --set-string secrets.databaseUrl="$(kubectl get secret db-creds -n prod -o jsonpath='{.data.url}' | base64 -d)" \\
  --atomic \\
  --timeout 15m

# 查看各环境部署状态
helm list -A | grep pipeline
"""

print("=== 练习 4 参考答案 ===")
print("\n--- values.yaml (dev defaults) ---")
print(values_base)
print("\n--- values-staging.yaml ---")
print(values_staging)
print("\n--- values-prod.yaml ---")
print(values_prod)
print("\n--- Deploy Commands ---")
print(deploy_commands)

---
# 复习要点

## Docker

- **Layer 缓存核心原则**: 变化频率低的指令放前面（FROM → apt-get → requirements.txt → pip install → COPY src/）
- **多阶段构建**: 分离 Build Stage（含编译工具、测试）和 Runtime Stage（精简）；只复制必要产物到 Runtime
- **安全最佳实践**: 非 root 用户运行（`useradd` + `USER appuser`），最小化系统依赖，清理 apt 缓存
- **.dockerignore**: 排除 .git、测试文件、文档、临时文件；减少构建上下文大小和意外泄漏

## Kubernetes

- **对象层级**: Namespace > Deployment > ReplicaSet > Pod > Container
- **Service 类型**: ClusterIP（内部）/ NodePort（节点端口）/ LoadBalancer（云负载均衡，生产常用）
- **PVC AccessMode**: ReadWriteOnce（单节点，大多数场景）/ ReadWriteMany（多节点共享，需 NFS/EFS）
- **Request vs Limit**:
  - CPU 超 Limit → **Throttle**（变慢，不 kill）
  - Memory 超 Limit → **OOM Kill**（Exit 137，直接 kill）
  - Request = Limit → **Guaranteed QoS**（最不会被驱逐）
- **KubernetesExecutor**: 每任务一个 Pod，完美隔离；冷启动 2-10s，不适合高频短任务
- **LimitRange**: 为容器设置默认值和边界上限（Namespace 级别）
- **ResourceQuota**: 限制整个 Namespace 的资源总量

## Helm

- **Chart 三要素**: Chart.yaml（元数据）/ values.yaml（默认参数）/ templates/（K8s YAML 模板）
- **值覆盖优先级**: `--set` > `-f values-override.yaml` > `values.yaml`（默认）
- **`--atomic`**: 升级失败自动回滚到上一个版本（生产必用）
- **`helm template`**: 本地渲染 YAML 不部署，用于 debug 和 GitOps

## 面试高频知识点

1. Docker Layer 缓存失效的传播方式（一层失效，之后全部失效）
2. CPU throttle vs Memory OOM Kill 的区别（内存超限会直接 kill，CPU 不会）
3. K8s QoS 三个等级及驱逐顺序（BestEffort → Burstable → Guaranteed）
4. Spark on K8s：executor 数量 = min(total_cpu/exec_cpu, total_memory/exec_memory)
5. ReadWriteMany vs ReadWriteOnce：Airflow DAG 共享需要 RWX + EFS/NFS